<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Gradient Descent Code-Along - Solution

---

### About
Understand and implement gradient descent, and recognize common pitfalls such as poor convergence and address them.

### Learning Objectives
- Understand the intuition behind gradient descent and how it solves optimization problems.
- Implement gradient descent.
- Understand common pitfalls associated with gradient descent.
- Identify solutions for common pitfalls.

### Notebook Guide
- Imports
- Let's see if we can do OLS by Gradient Descent!
- Putting it all together...

# Imports
Let's walk through how gradient descent works using code.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# The objective function

def f(x):
    return -np.log(x) / (1 + x)

In [ ]:
# Derivative of the objective function

def f_deriv(x):
    return -(1 + 1/x - np.log(x)) / (1 + x)**2

In [ ]:
# Let's see what the function looks like

xs = np.linspace(1, 7, 1000)
plt.plot(xs, f(xs));

In [ ]:
# Initial value and learning rate
x = 1
alpha = 1

# Iterate and apply gradient descent
x_steps = [x]
for i in range(300):
    x = x - alpha * f_deriv(x)
    x_steps.append(x)
    print(f"{i}: {x}")

In [ ]:
# Plot iterations

plt.plot(x_steps);

# Let's see if we can do OLS by Gradient Descent!

In [ ]:
# Set a random seed
np.random.seed(42)

In [ ]:
# Randomly generate data from a Poisson(45) distribution

temp = np.random.poisson(45, 100)

In [ ]:
# View array

temp

In [ ]:
# Calculate mean and sample variance of array

print(np.mean(temp))
print(np.var(temp, ddof = 1))

**US Ohio State Fun Facts:**
1. Ohio Stadium can seat 104,944 people. (Source: [Wikipedia](https://en.wikipedia.org/wiki/Ohio_Stadium).)
2. Ohio Stadium's record attendance is 110,045 people. (Source: [Wikipedia](https://en.wikipedia.org/wiki/Ohio_Stadium).)
3. Ohio State is better than Michigan. (Source: It's just a fact.)
4. Ohio State students enjoy soda. (Source: first-hand knowledge.)

In [ ]:
# sodas ~ N(200000 + 1000 * temp, 20000)

sodas_sold = 200000 + 1000 * temp + np.round(np.random.normal(0, 20000, 100))

In [ ]:
sodas_sold

$$
\text{sodas\_sold}_i = 200000 + 1000 \times \text{temp}_i + \varepsilon_i
$$

In [ ]:
# Create dataframe with temp and sodas_sold

df = pd.DataFrame({'temp': temp,
                   'sodas': sodas_sold})

In [ ]:
# Check the first five rows

df.head()

#### Our goal is to fit a model here.
- You and I know that our $y$-intercept $\beta_0$ is 200,000.
- You and I know that our slope $\beta_1$ is 1,000.
- However, our computer does not know that. Our computer has to estimate $\hat{\beta}_0$ and $\hat{\beta}_1$ from the data.
    - We might say that our **machine** has to... **learn**.

#### Our workflow:
1. Instantiate model.
2. Select a learning rate $\alpha$.
3. Select a starting point $\hat{\beta}_{1,0}$.
4. Calculate the gradient of the loss function.
5. Calculate $\hat{\beta}_{1,i+1} = \hat{\beta}_{1,i} - \alpha * \frac{\partial L}{\partial \beta_1}$.
6. Check value of $\left|\hat{\beta}_{1,i+1} - \hat{\beta}_{1,i}\right|$.
7. Repeat steps 4 through 6 until "stopping condition" is met.

#### Step 1. Instantiate model.

Our model takes on the form:
$$ Y = \beta_0 + \beta_1 X + \varepsilon$$

#### Step 2. Select a learning rate $\alpha$.

$$\alpha = 0.1$$

In [ ]:
alpha = 0.1

#### Step 3. Select a starting point.
The zero-th iteration of $\hat{\beta}_1$ is going to start at, say, 20.
$$\hat{\beta}_{1,0} = 20$$

Two points:
- You and I know that the true value of $\beta_1$ is 1000. We need the computer to figure (machine to learn) that part out!
- We're going to pretend like the computer already knows the value for $\beta_0$. In reality, we'd have to do this for $\beta_0$ and for $\beta_1$ at the same time.

In [ ]:
beta_1 = 20

#### Step 4. Calculate the gradient of the loss function with respect to parameter $\beta_1$.

The loss function, $L$, is our mean square error.

$$L = \frac{1}{n}\sum_{i = 1} ^ n (y_i - \hat{y}_i)^2 $$

$$\Rightarrow L = \frac{1}{n}\sum_{i = 1} ^ n \left(y_i - \left(\hat{\beta}_0 + \hat{\beta}_1x_i\right)\right)^2 $$

The gradient of this loss function with respect to $\beta_1$ is:

$$\frac{\partial L}{\partial \beta_1} = \frac{2}{n} \sum_{i=1}^n -x_i\left(y_i - \left(\hat{\beta}_1x_i + \hat{\beta}_0\right)\right) $$

In [ ]:
# Calculate gradient of beta_1

# One way to do it - suboptimal.
# def beta_1_gradient(x, y, beta_1, beta_0):
#     n = len(x)
#     # Start gradient at 0.
#     gradient = 0
#     # Begin summation.
#     for i in range(n):
#         # Add gradient for each observation.
#         gradient += -1 * x[i] * (y[i] - (beta_1 * x[i] + beta_0))
#     # Multiply gradient by 2 / n.
#     gradient *= (2 / n)
#     return gradient

# A better way to do it, using numpy

def beta_1_gradient(x, y, beta_1, beta_0):
    grads = -x * (y - (beta_1*x + beta_0))
    return 2 * np.mean(grads)

#### Step 5. Calculate $\hat{\beta}_{1,i+1} = \hat{\beta}_{1,i} - \alpha * \frac{\partial L}{\partial \beta_1}$.

In [ ]:
# Define function to calculate new value of beta_1

def update_beta_1(beta_1, alpha, gradient):
    beta_1 = beta_1 - alpha * gradient
    return beta_1

#### Step 6. Check value of $\left|\hat{\beta}_{1,i+1} - \hat{\beta}_{1,i}\right|$.

In [ ]:
def check_update(beta_1, updated_beta_1, tolerance = 0.1):
    return abs(beta_1 - updated_beta_1) < tolerance

#### Step 7: Save final value of $\hat{\beta}_1$.
See below

# Putting it all together...

In [ ]:
def gradient_descent(x, y, beta_1 = 0, alpha = 0.01, max_iter = 100):
    # Set converged = False
    converged = False
    
    # Iterate through our observations.
    step = 0
    while not converged:
        
        # Calculate gradient
        gradient = beta_1_gradient(x, y, beta_1, 200000)
        
        # Update beta_1
        updated_beta_1 = update_beta_1(beta_1, alpha, gradient)
        
        # Check for convergence
        converged = check_update(beta_1, updated_beta_1)
        
        # Overwrite beta_1
        beta_1 = updated_beta_1
        
        # Print out current step findings
        print(f'Iteration {step} with beta_1 value of {beta_1}.')
        
        # If we've converged, let us know!
        if converged:
            print(f'Our algorithm converged after {step} iterations with a beta_1 value of {beta_1}.')
        else:
            step += 1
            
        # If we exceed our step limit, break!
        if step > max_iter:
            break
        
    # If we didn't converge by the end of our loop, let us know!
    if not converged:
        print("Our algorithm did not converge, so do not trust the value of beta_1.")
    
    # Return beta_1
    return beta_1

In [ ]:
# Call gradient_descent with an initial beta_1 of 20, alpha of 0.01, and 100 iterations

gradient_descent(df['temp'],
                 df['sodas'],
                 beta_1 = 20,
                 alpha = 0.01,
                 max_iter = 100)

<details><summary>What should we do?</summary>

- We **should not** adjust our maximum iterations. It doesn't look like we'll converge.
- We should adjust our alpha!
</details>

In [ ]:
gradient_descent(df['temp'],
                 df['sodas'],
                 beta_1 = 20,
                 alpha = 0.0001,
                 max_iter = 100)